Importing the necessary libraries for dataset construction

In [118]:
import pandas as pd
import numpy as np
import tqdm
import boto3
from pokerkit import HandHistory
import io

AWS Constants

In [108]:
S3_BUCKET_NAME = 'pokeraidataset'
S3_PREFIX = ''
AWS_REGION = 'us-east-2'
client = boto3.client('s3', region_name = AWS_REGION)
resource = boto3.resource('s3', region_name = AWS_REGION)
paginator = client.get_paginator('list_objects_v2')

Testing getting and reading objects from S3

In [119]:
test_key = 'ps NLH handhq_1-OBFUSCATED.phhs'
response = client.get_object(Bucket = S3_BUCKET_NAME, Key = test_key)
hands_bytes = response['Body'].read()
hands_example = hands_bytes.decode('utf-8')
print(hands_example)

[1]
variant = 'NT'
ante_trimming_status = false
antes = [0, 0, 0, 0, 0, 0]
blinds_or_straddles = [0.25, 0.50, 0, 0, 0, 0]
min_bet = 0.50
starting_stacks = [40.90, 42.95, 17.85, 22.60, 187.45, 70.85]
actions = ['d dh p1 ????', 'd dh p2 ????', 'd dh p3 ????', 'd dh p4 ????', 'd dh p5 ????', 'd dh p6 ????', 'p3 f', 'p4 f', 'p5 f', 'p6 cbr 2.00', 'p1 f', 'p2 f']
venue = 'PokerStars'
time = 00:00:00
day = 1
month = 7
year = 2009
hand = 59937793578
seats = [6, 1, 2, 3, 4, 5]
seat_count = 6
table = 'QOgaEqMcJ73pcUMMxoisUg'
players = ['s83dnhZ6VC53vY0/f6OV/g', 'RFefXIDyFvNtJLq+CloCzA', 'qsObMEIlRO22OlS867m/lw', 'wUZNozoIDYf+16hp4nn6dw', 'gdb5VNZ4Chf1Hioo7N/jsA', 'HoaSQCRso+nEyO5DUhGduw']
winnings = [0, 0, 0, 0, 0, 0]
currency_symbol = '$'
time_zone_abbreviation = 'ET'

[2]
variant = 'NT'
ante_trimming_status = false
antes = [0, 0, 0, 0, 0, 0, 0]
blinds_or_straddles = [0.25, 0.50, 0, 0, 0, 0, 0]
min_bet = 0.50
starting_stacks = [16.50, 43.65, 30.90, 31.80, 9.75, 13.35, 123]
actions = ['d dh p1 

Function to be able to parse .phh files

In [ ]:
def parse_phhs(s3_body):
    """
    Parse a poker hand histories file (.phhs) from a S3 Body and returns a list of HandHistory objects.
    """
    hands = []
    bytes = io.BytesIO(s3_body.read())
    hhs = list(HandHistory.load_all(bytes))

    for hh in hhs:
        hands.append(hh)
    return hhs


[HandHistory(variant='NT', ante_trimming_status=False, antes=[0, 0, 0, 0, 0, 0], blinds_or_straddles=[Decimal('0.25'), Decimal('0.50'), 0, 0, 0, 0], bring_in=None, small_bet=None, big_bet=None, min_bet=Decimal('0.50'), starting_stacks=[Decimal('40.90'), Decimal('42.95'), Decimal('17.85'), Decimal('22.60'), Decimal('187.45'), Decimal('70.85')], actions=['d dh p1 ????', 'd dh p2 ????', 'd dh p3 ????', 'd dh p4 ????', 'd dh p5 ????', 'd dh p6 ????', 'p3 f', 'p4 f', 'p5 f', 'p6 cbr 2.00', 'p1 f', 'p2 f'], author=None, event=None, url=None, venue='PokerStars', address=None, city=None, region=None, postal_code=None, country=None, time=datetime.time(0, 0), time_zone=None, day=1, month=7, year=2009, hand=59937793578, level=None, seats=[6, 1, 2, 3, 4, 5], seat_count=6, table='QOgaEqMcJ73pcUMMxoisUg', players=['s83dnhZ6VC53vY0/f6OV/g', 'RFefXIDyFvNtJLq+CloCzA', 'qsObMEIlRO22OlS867m/lw', 'wUZNozoIDYf+16hp4nn6dw', 'gdb5VNZ4Chf1Hioo7N/jsA', 'HoaSQCRso+nEyO5DUhGduw'], finishing_stacks=None, winnings

c:\Users\jonat\repos\PokerAI\.venv\Lib\site-packages\pokerkit\notation.py:434: UserWarning: The field 'time_zone_abbreviation' is an unexpected field and should probably be prefixed with an underscore character '_'.
  warn(


Testing parse_phhs function

In [130]:
test_response = client.get_object(Bucket=S3_BUCKET_NAME, Key=test_key)
h = parse_phhs(test_response['Body'])
print(h[0:5])
print(f"Successfully parsed {len(h)} hands.")

[HandHistory(variant='NT', ante_trimming_status=False, antes=[0, 0, 0, 0, 0, 0], blinds_or_straddles=[Decimal('0.25'), Decimal('0.50'), 0, 0, 0, 0], bring_in=None, small_bet=None, big_bet=None, min_bet=Decimal('0.50'), starting_stacks=[Decimal('40.90'), Decimal('42.95'), Decimal('17.85'), Decimal('22.60'), Decimal('187.45'), Decimal('70.85')], actions=['d dh p1 ????', 'd dh p2 ????', 'd dh p3 ????', 'd dh p4 ????', 'd dh p5 ????', 'd dh p6 ????', 'p3 f', 'p4 f', 'p5 f', 'p6 cbr 2.00', 'p1 f', 'p2 f'], author=None, event=None, url=None, venue='PokerStars', address=None, city=None, region=None, postal_code=None, country=None, time=datetime.time(0, 0), time_zone=None, day=1, month=7, year=2009, hand=59937793578, level=None, seats=[6, 1, 2, 3, 4, 5], seat_count=6, table='QOgaEqMcJ73pcUMMxoisUg', players=['s83dnhZ6VC53vY0/f6OV/g', 'RFefXIDyFvNtJLq+CloCzA', 'qsObMEIlRO22OlS867m/lw', 'wUZNozoIDYf+16hp4nn6dw', 'gdb5VNZ4Chf1Hioo7N/jsA', 'HoaSQCRso+nEyO5DUhGduw'], finishing_stacks=None, winnings

Loading all .phh files from S3

In [ ]:
hands = []

response_iterator = paginator.paginate(
    Bucket = S3_BUCKET_NAME,
)

for page in response_iterator:
    if 'Contents' in page:
        for obj in page['Contents']:
            obj_key = obj['Key']
            hands.append(client.get_object(Bucket = S3_BUCKET_NAME, Key = obj_key)['Body'])
print(f'Found {len(hands)} hands in the dataset.')
print(hands[0:2])